In [56]:
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [58]:
%sql sqlite:///travel.sqlite

When the airport codes are put together, they form the route or flight path. Grouping by both the departure and arrival cities, we can see which combinations have the most amount of instances.

In [60]:
%%sql
SELECT departure_airport,arrival_airport, COUNT(*) AS flights
FROM flights
GROUP BY departure_airport, arrival_airport
order by flights desc
limit 10;

 * sqlite:///travel.sqlite
Done.


departure_airport,arrival_airport,flights
LED,SVO,305
SVO,LED,305
DME,LED,244
LED,DME,244
BZK,DME,183
BZK,SVO,183
BZK,VKO,183
DME,BZK,183
LED,VKO,183
SVO,BZK,183


By joining 3 tables - flights, airports_data (aliased as f) and airports_data (aliased as t), we can retrieve flight details along with origin and destination airport cities and coordinates.

In [61]:
%%sql
SELECT flights.flight_id, f.city as from_city, f.coordinates as from_coordinates, t.city as to_city, t.coordinates as to_coordinates
from flights
left join airports_data as f
on flights.departure_airport = f.airport_code
left join airports_data as t
on flights.arrival_airport = t.airport_code
limit 10;

 * sqlite:///travel.sqlite
Done.


flight_id,from_city,from_coordinates,to_city,to_coordinates
1185,"{""en"": ""Moscow"", ""ru"": ""Москва""}","(37.9062995910644531,55.4087982177734375)","{""en"": ""Bratsk"", ""ru"": ""Братск""}","(101.697998046875,56.3706016540527344)"
3979,"{""en"": ""Moscow"", ""ru"": ""Москва""}","(37.2615013122999983,55.5914993286000012)","{""en"": ""Khanty-Mansiysk"", ""ru"": ""Ханты-Мансийск""}","(69.0860977172851562,61.0284996032714844)"
4739,"{""en"": ""Moscow"", ""ru"": ""Москва""}","(37.2615013122999983,55.5914993286000012)","{""en"": ""Sochi"", ""ru"": ""Сочи""}","(39.9566001892089986,43.4499015808110016)"
5502,"{""en"": ""Moscow"", ""ru"": ""Москва""}","(37.4146000000000001,55.9725990000000024)","{""en"": ""Ufa"", ""ru"": ""Уфа""}","(55.8744010925289984,54.5574989318850001)"
6938,"{""en"": ""Moscow"", ""ru"": ""Москва""}","(37.4146000000000001,55.9725990000000024)","{""en"": ""Ulyanovsk"", ""ru"": ""Ульяновск""}","(48.2266998291000064,54.2682991027999932)"
7784,"{""en"": ""Moscow"", ""ru"": ""Москва""}","(37.4146000000000001,55.9725990000000024)","{""en"": ""Kurgan"", ""ru"": ""Курган""}","(65.4156036376953125,55.4752998352050781)"
9478,"{""en"": ""St. Petersburg"", ""ru"": ""Санкт-Петербург""}","(30.2625007629394531,59.8003005981445312)","{""en"": ""Orenburg"", ""ru"": ""Оренбург""}","(55.4566993713378906,51.7957992553710938)"
11085,"{""en"": ""Yekaterinburg"", ""ru"": ""Екатеринбург""}","(60.8027000427250002,56.7430992126460012)","{""en"": ""Syktyvkar"", ""ru"": ""Сыктывкар""}","(50.8451004028320312,61.6469993591308594)"
11847,"{""en"": ""Kazan"", ""ru"": ""Казань""}","(49.278701782227003,55.606201171875)","{""en"": ""Irkutsk"", ""ru"": ""Иркутск""}","(104.388999938959998,52.2680015563960012)"
12012,"{""en"": ""Kazan"", ""ru"": ""Казань""}","(49.278701782227003,55.606201171875)","{""en"": ""Magnetiogorsk"", ""ru"": ""Магнитогорск""}","(58.7556991577148438,53.3931007385253906)"


As you can see, the data inside those rows is in JSONs and needs to be extracted to used. I am going to store the table with the extracted information in a temporary table to use throughout the code.

In [62]:
%%sql
CREATE TEMP TABLE FLIGHT_INFO AS
  SELECT flights.flight_id, json_extract(departure.city, '$.en') AS from_city,
    CAST(SUBSTR(departure.coordinates, 2, INSTR(departure.coordinates, ',') - 2) AS REAL) AS from_longitude,
    CAST(SUBSTR(departure.coordinates, INSTR(departure.coordinates, ',') + 1, LENGTH(departure.coordinates) - INSTR(departure.coordinates, ',') - 2) AS REAL) AS from_latitude,
    json_extract(arrival.city, '$.en') AS to_city,
    CAST(SUBSTR(arrival.coordinates, 2, INSTR(arrival.coordinates, ',') - 2) AS REAL) AS to_longitude,
    CAST(SUBSTR(arrival.coordinates, INSTR(arrival.coordinates, ',') + 1, LENGTH(arrival.coordinates) - INSTR(arrival.coordinates, ',') - 2) AS REAL) AS to_latitude
  FROM flights
  LEFT JOIN airports_data as departure ON flights.departure_airport = departure.airport_code
  LEFT JOIN airports_data as arrival ON flights.arrival_airport = arrival.airport_code;

  CREATE INDEX idx_flight_id ON FLIGHT_INFO (flight_id);

 * sqlite:///travel.sqlite
Done.
Done.


[]

In [64]:
%%sql
SELECT * FROM FLIGHT_INFO
LIMIT 10;

 * sqlite:///travel.sqlite
Done.


flight_id,from_city,from_longitude,from_latitude,to_city,to_longitude,to_latitude
1185,Moscow,37.90629959106445,55.40879821777344,Bratsk,101.697998046875,56.370601654052734
3979,Moscow,37.2615013123,55.5914993286,Khanty-Mansiysk,69.08609771728516,61.028499603271484
4739,Moscow,37.2615013123,55.5914993286,Sochi,39.956600189209,43.449901580811
5502,Moscow,37.4146,55.972599,Ufa,55.874401092529,54.557498931885
6938,Moscow,37.4146,55.972599,Ulyanovsk,48.226699829100006,54.26829910279999
7784,Moscow,37.4146,55.972599,Kurgan,65.41560363769531,55.47529983520508
9478,St. Petersburg,30.262500762939453,59.80030059814453,Orenburg,55.45669937133789,51.795799255371094
11085,Yekaterinburg,60.802700042725,56.743099212646,Syktyvkar,50.84510040283203,61.64699935913086
11847,Kazan,49.278701782227,55.60620117187,Irkutsk,104.38899993896,52.268001556396
12012,Kazan,49.278701782227,55.60620117187,Magnetiogorsk,58.755699157714844,53.39310073852539


From the 'FLIGHT_INFO' table above, we can calculate the average distance between two cities using the Haversine formula.

In [69]:
%%sql
SELECT flight_id, from_city, to_city, ROUND(AVG(distance_km), 1) AS average_distance_km
FROM (
    SELECT flight_id,  from_city, to_city,
        2 * 6371 * ASIN(SQRT(
        POWER(SIN(RADIANS((to_latitude - from_latitude) / 2)), 2) +
        COS(RADIANS(from_latitude)) * COS(RADIANS(to_latitude)) *
        POWER(SIN(RADIANS((to_longitude - from_longitude) / 2)), 2)
        )) AS distance_km
    FROM FLIGHT_INFO
) AS subquery
GROUP BY from_city, to_city, flight_id
ORDER BY average_distance_km DESC
LIMIT 10;

 * sqlite:///travel.sqlite
Done.


flight_id,from_city,to_city,average_distance_km
1308,Moscow,Petropavlovsk,6774.7
1309,Moscow,Petropavlovsk,6774.7
1310,Moscow,Petropavlovsk,6774.7
1311,Moscow,Petropavlovsk,6774.7
1312,Moscow,Petropavlovsk,6774.7
1313,Moscow,Petropavlovsk,6774.7
1314,Moscow,Petropavlovsk,6774.7
1315,Moscow,Petropavlovsk,6774.7
1316,Moscow,Petropavlovsk,6774.7
1317,Moscow,Petropavlovsk,6774.7
